In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .replace("<s>","")
                .replace("</s>","")
                .replace("<pad>","")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [5]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/project_bpe_16k.pkl"
train_path = "/kaggle/input/datasets/ksaiveerendraprasad/mha-kernel-results/train_tokens2.npy"
val_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/val_tokens1 (1).npy"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    bpe = pickle.load(f)

# Load token arrays
train_tokens = np.load(train_path, mmap_mode="r")
val_tokens = np.load(val_path, mmap_mode="r")

print("BPE vocab size:", len(bpe.itos))
print("Train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))

BPE vocab size: 7999
Train tokens: 200000000
Validation tokens: 5000000


In [6]:
from typing import List
class TokenDataset(torch.utils.data.Dataset):
    """Fixed-length chunks of a tokenized corpus for next-token-prediction training."""

    def __init__(self, token_ids: List[int], seq_len: int):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return max(0, (len(self.ids) - 1) // self.seq_len)

    def __getitem__(self, i):
        start = i * self.seq_len
        x = self.ids[start : start + self.seq_len]
        y = self.ids[start + 1 : start + self.seq_len + 1]
        return x, y

In [7]:
BATCH_SIZE=16

In [8]:
#prepare data and val loader

train_dataset  = TokenDataset(train_tokens, MAX_SEQ_LEN)
val_dataset = TokenDataset(val_tokens, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [9]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def chunked_gqa_linear_attn(q, k, v, chunk=128, eps=1e-6):
    """
    q: (B, Hq, T, F)   k: (B, Hkv, T, F)   v: (B, Hkv, T, D)   all float32, features >= 0
    Replaces the kv / cumsum / einsum block. Returns out (B, Hq, T, D) and final state (S, z).
    """
    B, Hq, T, Fd = q.shape
    Hkv, D = k.shape[1], v.shape[-1]
    R = Hq // Hkv
    q = q.view(B, Hkv, R, T, Fd)                      # query head h uses KV head h // R

    S = q.new_zeros(B, Hkv, Fd, D)                    # running sum of phi(k)^T v
    z = q.new_zeros(B, Hkv, Fd)                       # running sum of phi(k)
    tri = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))

    outs = []
    for s in range(0, T, chunk):
        qc = q[:, :, :, s:s + chunk]                  # (B,Hkv,R,c,F)
        kc = k[:, :, s:s + chunk]                     # (B,Hkv,c,F)
        vc = v[:, :, s:s + chunk]                     # (B,Hkv,c,D)
        c = qc.shape[3]

        a = (qc @ kc.unsqueeze(2).transpose(-1, -2)).masked_fill(~tri[:c, :c], 0.0)  # inside chunk
        num = qc @ S.unsqueeze(2) + a @ vc.unsqueeze(2)                             # past + current
        den = (qc * z[:, :, None, None, :]).sum(-1, keepdim=True) + a.sum(-1, keepdim=True)
        outs.append(num / (den + eps))

        S = S + kc.transpose(-1, -2) @ vc             # update state after the chunk
        z = z + kc.sum(dim=2)

    return torch.cat(outs, dim=3).reshape(B, Hq, T, D), (S, z)


class LinearAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.0, feat_dim=32, chunk=128):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.chunk = chunk
        self.feat_dim = feat_dim
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        # kernel: phi(x) = (gamma * norm(x) + beta)^2  (gamma, beta in the LayerNorm)
        self.q_kernel = nn.Linear(self.d_head, feat_dim, bias=True)
        self.k_kernel = nn.Linear(self.d_head, feat_dim, bias=True)
        self.q_norm = nn.LayerNorm(feat_dim)
        self.k_norm = nn.LayerNorm(feat_dim)

        self.resid_dropout = nn.Dropout(dropout)

    def _project(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        q_phi = self.q_norm(self.q_kernel(q)).float() ** 2     # (B,n_heads,T,F)
        k_phi = self.k_norm(self.k_kernel(k)).float() ** 2     # (B,n_kv,T,F) 
        return q_phi, k_phi, v.float()

    def forward(self, x, return_state=False):
        B, T, C = x.shape
        q_phi, k_phi, v = self._project(x)
        with torch.autocast(device_type="cuda", enabled=False):
            out, state = chunked_gqa_linear_attn(q_phi, k_phi, v, self.chunk)
        out = F.rms_norm(out.to(x.dtype), (self.d_head,))
        out = self.out_proj(out.transpose(1, 2).reshape(B, T, C))
        out = self.resid_dropout(out)
        return (out, state) if return_state else out

    @torch.no_grad()
    def step(self, x_t, S, z, eps=1e-6):
        """One-token decoding. x_t: (B,1,C), S: (B,Hkv,F,D), z: (B,Hkv,F), both float32."""
        B = x_t.shape[0]
        q_phi, k_phi, v = self._project(x_t)
        S = S + k_phi.transpose(-1, -2) @ v
        z = z + k_phi.squeeze(2)
    
        qg = q_phi.reshape(
            B, self.n_kv_heads, self.n_rep, self.feat_dim
        )
    
        numerator = torch.einsum("bhrf,bhfd->bhrd", qg, S)
        denominator = (qg * z.unsqueeze(2)).sum(-1, keepdim=True)
    
        out = numerator / (denominator + eps)
        out = F.rms_norm(out.to(x_t.dtype), (self.d_head,))
        out = out.reshape(B, 1, self.n_heads * self.d_head)
        out = self.out_proj(out)
    
        return out, S, z

In [10]:

"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    feat_dim: int = 32                 # kernel feature dimension F
    chunk: int = 128 


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)



class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = LinearAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads,
                                       cfg.dropout, cfg.feat_dim, cfg.chunk)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, return_state=False):
        if return_state:
            a, state = self.attn(self.ln1(x), return_state=True)
        else:
            a, state = self.attn(self.ln1(x)), None
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return (x, state) if return_state else x

    @torch.no_grad()
    def step(self, x_t, state):
        S, z = state
        a, S, z = self.attn.step(self.ln1(x_t), S, z)
        x = x_t + a
        x = x + self.ffn(self.ln2(x))
        return x, (S, z)

def sinusoidal_positional_encoding(max_len, d_model):
    pos = torch.arange(max_len, dtype=torch.float32)[:, None]
    i = torch.arange(d_model // 2, dtype=torch.float32)[None, :]
    ang = pos / (10000 ** (2 * i / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(ang), torch.cos(ang)
    return pe


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                                 persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        with torch.no_grad():
                nn.init.uniform_(
                    self.tok_emb.weight,
                    -1 / math.sqrt(cfg.d_model),
                    1 / math.sqrt(cfg.d_model)
                )

    def _embed(self, idx, start=0):
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            T = idx.shape[1]
            x = x + self.pos_emb[start:start + T]
        return x

    def forward(self, idx):
        x = self._embed(idx)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

    # ---------- inference ----------
    @torch.no_grad()
    def prefill(self, idx):
        x = self._embed(idx)
        states = []
        for block in self.blocks:
            x, st = block(x, return_state=True)
            states.append(st)
        return self.head(self.ln_f(x[:, -1:])), states          # logits of last token

    @torch.no_grad()
    def step(self, tok, pos, states):
        x = self._embed(tok, start=pos)                          # tok: (B,1)
        new = []
        for block, st in zip(self.blocks, states):
            x, st = block.step(x, st)
            new.append(st)
        return self.head(self.ln_f(x)), new

In [11]:
@torch.no_grad()
def perplexity_of_loader(model, loader, device=device):
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for i, (src, target) in enumerate(loader):
        src, target = src.to(device), target.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(src)
        loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                               target.reshape(-1), reduction="sum")
        total_nll += loss.item()
        total_tokens += target.numel()
    return math.exp(total_nll / total_tokens) if total_tokens else float("inf")

In [12]:
# import time, math

# def get_lr(step, max_steps, peak_lr, warmup, min_ratio=0.1):
#     if step < warmup:
#         return peak_lr * (step + 1) / warmup
#     p = min(1.0, (step - warmup) / max(1, max_steps - warmup))
#     return peak_lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p)))


# def train(train_loader, val_loader, model, optimizer, device=device,
#           train_steps=1000, N=100, peak_lr=6e-4, warmup=100, grad_clip=1.0, 
#           ckpt_path="linear_gqa_best.pt", val_batches=50):
#     scaler = torch.amp.GradScaler("cuda")   # fp16 on T4; skip for bf16
#     hist = {"train_loss": [], "held_out_ppl": [], "grad_norm": []}
#     best_ppl, step = float("inf"), 0
#     start = time.time()
#     accum = 4                                   
#     micro = 0
#     optimizer.zero_grad(set_to_none=True)
#     model.train()
#     for src, target in train_loader:            
#             src, target = src.to(device), target.to(device)
#             with torch.autocast(device_type="cuda", dtype=torch.float16):
#                 logits = model(src)
#             loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
#                                    target.reshape(-1)) / accum
#             if not torch.isfinite(loss):
#                 raise RuntimeError(f"Non-finite loss at step {step}; lower the LR or check the kernel.")
#             scaler.scale(loss).backward()
#             micro += 1
#             if micro % accum:                       # not yet time to step
#                 continue
        
#             for g in optimizer.param_groups:
#                 g["lr"] = get_lr(step, train_steps, peak_lr, warmup)
#             scaler.unscale_(optimizer)
#             gn = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
#             scaler.step(optimizer); scaler.update()
#             optimizer.zero_grad(set_to_none=True)
#             step += 1
#             hist["train_loss"].append((step, loss.item() * accum))
#             hist["grad_norm"].append((step, gn.item()))

            
#             if step % N == 0:
#                 ppl = perplexity_of_loader(model, val_loader, device)
#                 hist["held_out_ppl"].append((step, ppl))
#                 print(f"Step {step}/{train_steps} | loss {loss.item():.4f} | "
#                       f"val ppl {ppl:.3f} | gnorm {gn.item():.2f} | "
#                       f"lr {optimizer.param_groups[0]['lr']:.2e} | {(time.time()-start)/60:.1f} min")
#                 if ppl < best_ppl:
#                     best_ppl = ppl
#                     torch.save({"step": step, "model": model.state_dict(),
#                                 "optimizer": optimizer.state_dict(),
#                                 "best_ppl": best_ppl}, ckpt_path)
#                 model.train()

#             if step >= train_steps:
#                 break

#     ck = torch.load(ckpt_path, map_location=device)          # reload best
#     model.load_state_dict(ck["model"])
#     print(f"\nDone in {(time.time()-start)/60:.1f} min | best val ppl {best_ppl:.3f}")
#     return hist, best_ppl
import time
import math

def get_lr(step, max_steps, peak_lr, warmup, min_ratio=0.1):
    if step < warmup:
        return peak_lr * (step + 1) / warmup

    p = min(1.0, (step - warmup) / max(1, max_steps - warmup))

    return peak_lr * (
        min_ratio
        + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p))
    )


def train(
    train_loader,
    val_loader,
    model,
    optimizer,
    device=device,
    train_steps=1000,
    N=100,
    peak_lr=6e-4,
    warmup=100,
    grad_clip=1.0,
    ckpt_path="kernel_best.pt",
    val_batches=50
):
    scaler = torch.amp.GradScaler("cuda")

    hist = {
        "train_loss": [],
        "held_out_ppl": [],
        "grad_norm": []
    }

    best_ppl = float("inf")
    step = 0
    start = time.time()

    optimizer.zero_grad(set_to_none=True)
    model.train()

    while step < train_steps:
        for src, target in train_loader:
            src = src.to(device)
            target = target.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(src)

            loss = F.cross_entropy(
                logits.float().reshape(-1, logits.size(-1)),
                target.reshape(-1)
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at step {step}; "
                    "lower the LR or check the kernel."
                )

            scaler.scale(loss).backward()

            for g in optimizer.param_groups:
                g["lr"] = get_lr(
                    step, train_steps, peak_lr, warmup
                )

            scaler.unscale_(optimizer)

            gn = torch.nn.utils.clip_grad_norm_(
                model.parameters(), grad_clip
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            step += 1

            hist["train_loss"].append((step, loss.item()))
            hist["grad_norm"].append((step, gn.item()))

            if step % N == 0:
                ppl = perplexity_of_loader(
                    model, val_loader, device
                )

                hist["held_out_ppl"].append((step, ppl))

                print(
                    f"Step {step}/{train_steps} | "
                    f"loss {loss.item():.4f} | "
                    f"val ppl {ppl:.3f} | "
                    f"gnorm {gn.item():.2f} | "
                    f"lr {optimizer.param_groups[0]['lr']:.2e} | "
                    f"{(time.time() - start) / 60:.1f} min"
                )

                if ppl < best_ppl:
                    best_ppl = ppl

                    torch.save({
                        "step": step,
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "best_ppl": best_ppl
                    }, ckpt_path)

                model.train()

            if step >= train_steps:
                break

    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])

    print(
        f"\nDone in {(time.time() - start) / 60:.1f} min | "
        f"best val ppl {best_ppl:.3f}"
    )

    return hist, best_ppl

In [13]:
@torch.no_grad()
def generate_tokens(model, text, target_words=128, max_new_tokens=2048,
                    temperature=0.8, top_k=40, eos_id=3):
    model.eval()
    max_len = model.cfg.max_seq_len
    ids = bpe.encode(text)
    ids = ids[-(max_len - 1):]                                   # keep the prompt within limits
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    logits, states = model.prefill(x)                            # (1,1,V), per-layer state
    pos = x.shape[1]
    all_ids, generated, words = list(ids), [], 0

    for _ in range(max_new_tokens):
        l = logits[:, -1].float()
        if temperature == 0:
            nxt = l.argmax(-1, keepdim=True)
        else:
            l = l / temperature
            if top_k:
                v, _ = torch.topk(l, top_k)
                l = l.masked_fill(l < v[:, [-1]], float("-inf"))
            nxt = torch.multinomial(F.softmax(l, -1), 1)

        tid = nxt.item()
        if tid == eos_id:
            break
        generated.append(tid)
        all_ids.append(tid)
        if bpe.itos[tid].endswith(b"</w>"):
            words += 1
            if words >= target_words:
                break

        if pos >= max_len:                                       # position table exhausted
            ctx = torch.tensor(all_ids[-(max_len // 2):], device=device).unsqueeze(0)
            logits, states = model.prefill(ctx)
            pos = ctx.shape[1]
        else:
            logits, states = model.step(nxt, pos, states)
            pos += 1

    return bpe.decode(generated)

In [14]:
cfg_sin = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="sinusoidal",
    feat_dim=32,
    chunk=128,
    
)

raw_model = DecoderLM(cfg_sin).to(device)          # use this for generate / bench / save / load
model_sin = torch.nn.DataParallel(raw_model) if torch.cuda.device_count() > 1 else raw_model
decay    = [p for p in raw_model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay = [p for p in raw_model.parameters() if p.requires_grad and p.dim() < 2]

optimizer_sin = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=3e-4, betas=(0.9, 0.95), eps=1e-8,
)

In [15]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model_sin))

total parameters :  67142912


In [16]:
BATCH_SIZE

16

In [17]:
checkpoint = torch.load(
    "/kaggle/input/datasets/ksaiveerendraprasad/mha-kernel-results/results (12)/kernel_best.pt",
    map_location=device,
    weights_only=False
)

model_sin.load_state_dict(checkpoint["model"])

optimizer_sin.load_state_dict(checkpoint["optimizer"])

print("Best PPL:", checkpoint["best_ppl"])
print("Checkpoint step:", checkpoint["step"])

Best PPL: 7.903215571805735
Checkpoint step: 12400


In [18]:
hist, best_ppl = train(train_loader, val_loader, model_sin, optimizer_sin,
                   device=device, peak_lr=3e-4, warmup=150, train_steps=12500, N=100)

Step 100/12500 | loss 3.3821 | val ppl 9.377 | gnorm 48.33 | lr 2.00e-04 | 1.9 min
Step 200/12500 | loss 3.2361 | val ppl 11.216 | gnorm 16.24 | lr 3.00e-04 | 3.8 min
Step 300/12500 | loss 3.2330 | val ppl 10.590 | gnorm 3.98 | lr 3.00e-04 | 5.7 min
Step 400/12500 | loss 2.7107 | val ppl 10.453 | gnorm 5.47 | lr 3.00e-04 | 7.6 min
Step 500/12500 | loss 2.8683 | val ppl 10.480 | gnorm 6.58 | lr 2.99e-04 | 9.5 min
Step 600/12500 | loss 3.0445 | val ppl 10.463 | gnorm 3.96 | lr 2.99e-04 | 11.4 min
Step 700/12500 | loss 2.9641 | val ppl 10.224 | gnorm 4.69 | lr 2.99e-04 | 13.3 min
Step 800/12500 | loss 3.0986 | val ppl 10.108 | gnorm 4.22 | lr 2.98e-04 | 15.2 min
Step 900/12500 | loss 2.8891 | val ppl 10.013 | gnorm 3.41 | lr 2.98e-04 | 17.1 min
Step 1000/12500 | loss 3.0197 | val ppl 9.885 | gnorm 6.42 | lr 2.97e-04 | 19.0 min
Step 1100/12500 | loss 3.3277 | val ppl 9.664 | gnorm 3.66 | lr 2.96e-04 | 20.9 min
Step 1200/12500 | loss 2.6390 | val ppl 9.671 | gnorm 3.19 | lr 2.95e-04 | 22.8 

In [19]:
torch.save({
        "model_state_dict": {
        k: v.detach().cpu().clone()
        for k, v in model_sin.state_dict().items()
    },
        "optimizer_state_dict": optimizer_sin.state_dict(),
        "best_ppl": best_ppl,
         "history ":hist
    }, "kernel_final.pt")

In [20]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:")
print(test_file_path)
with open(test_file_path,"r", encoding="utf-8") as f:
    test_data = json.load(f)

test.json: 0.00B [00:00, ?B/s]

Test file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [21]:

predictions = {}

for  id, data in test_data.items():
    text =data["text"]
    input_ids = bpe.encode(text)
    print(id)
    print(f"  BPE length: {len(input_ids)}")
    if len(input_ids) == 0:
        print("EMPTY INPUT:", id)
        print("Text:", repr(text[:200]))
        continue
    predictions[id] = generate_tokens(model_sin.module, text)
    

1
  BPE length: 674
2
  BPE length: 666
3
  BPE length: 586
4
  BPE length: 637
5
  BPE length: 612
6
  BPE length: 619
7
  BPE length: 661
8
  BPE length: 722
9
  BPE length: 646
10
  BPE length: 612
11
  BPE length: 771
12
  BPE length: 897
13
  BPE length: 610
14
  BPE length: 670
15
  BPE length: 671
16
  BPE length: 675
17
  BPE length: 645
18
  BPE length: 710
19
  BPE length: 782
20
  BPE length: 662
21
  BPE length: 594
22
  BPE length: 596
23
  BPE length: 593
24
  BPE length: 660
25
  BPE length: 586
26
  BPE length: 650
27
  BPE length: 642
28
  BPE length: 637
29
  BPE length: 688
30
  BPE length: 670
31
  BPE length: 744
32
  BPE length: 635
33
  BPE length: 604
34
  BPE length: 603
35
  BPE length: 588
36
  BPE length: 673
37
  BPE length: 705
38
  BPE length: 725
39
  BPE length: 769
40
  BPE length: 739
41
  BPE length: 706
42
  BPE length: 616
43
  BPE length: 612
44
  BPE length: 582
45
  BPE length: 599
46
  BPE length: 634
47
  BPE length: 667
48
  BPE length: 657
4

In [22]:
special_tokens = ["<s>", "</s>", "<pad>"]

predictions = {
    id: pred.replace("<s>", "").replace("</s>", "").replace("<pad>", "").strip()
    for id, pred in predictions.items()
}

In [23]:
import json

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            id: {"text": text}
            for id, text in predictions.items()
        },
        f,
        ensure_ascii=False,
        indent=2
    )

In [24]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip
